In [2]:
# =================================================================================================
# NHANES 2015-2016
# SLEEP DISORDERS QUESTIONNAIRE - SLQ_I
#
# INPUT:
#   SLQ_I.xpt
#
# OUTPUT:
#   SLQ_I_corrected.csv
#
#
# COMPONENT:
#   Questionnaire
#
# SUBCOMPONENT:
#   Sleep Disorders
#
#
# ATTACHED SOURCE FINDINGS
# -------------------------------------------------------------------------------------------------
#
# Rows                         = 6,327
# Columns                      = 8
#
# Variables:
#
#   SEQN
#   SLQ300
#   SLQ310
#   SLD012
#   SLQ030
#   SLQ040
#   SLQ050
#   SLQ120
#
# SEQN range                   = 83732-93702
# Missing SEQN                 = 0
# Duplicate SEQN               = 0
#
#
# VALIDATED SAS XPORT TINY-ZERO ARTIFACTS
# -------------------------------------------------------------------------------------------------
#
# SLQ030     1,820
# SLQ040     4,535
# SLQ120     1,159
#
# TOTAL       7,514
#
#
# CHARACTER VARIABLES
# -------------------------------------------------------------------------------------------------
#
# SLQ300
# SLQ310
#
# These are character HH:MM fields.
#
# IMPORTANT:
#
# DO NOT require pandas dtype == object.
#
# Depending on pandas version these may load as:
#
#   object
#
# or:
#
#   string / StringDtype
#
# Fidelity is validated from actual text content instead.
#
#
# GENUINE DECIMAL VARIABLE
# -------------------------------------------------------------------------------------------------
#
# SLD012
#
# CDC:
#
#   Sleep duration calculated from sleep/wake times.
#   Values rounded to nearest half-hour by NHANES.
#   Range = 2 to 14.5
#
# DO NOT integerize or round SLD012.
#
#
# PROVEN INTEGRAL/CODE VARIABLES
# -------------------------------------------------------------------------------------------------
#
# SEQN
# SLQ030
# SLQ040
# SLQ050
# SLQ120
#
#
# FIDELITY RULES
# -------------------------------------------------------------------------------------------------
#
# 1. Restore ONLY validated SAS XPORT tiny-zero artifacts.
#
# 2. Never globally replace tiny values.
#
# 3. Preserve SLQ300 / SLQ310 character text exactly.
#
# 4. Preserve SLD012 as a genuine decimal variable.
#
# 5. Remove ".0" only from proven integral/code variables.
#
# 6. Do NOT round.
#
# 7. Do NOT use float_format=.
#
# 8. Preserve missing positions.
#
# 9. Validate CDC distributions.
#
# 10. Validate exact numeric XPT -> CSV round-trip.
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)


XPT_FILE = (
    DATA_DIR
    / "SLQ_I.xpt"
)


CSV_FILE = (
    DATA_DIR
    / "SLQ_I_corrected.csv"
)


# =================================================================================================
# 3. VERIFY INPUT FILE EXISTS
# =================================================================================================

if not XPT_FILE.exists():

    raise FileNotFoundError(
        f"SLQ_I XPT file not found:\n{XPT_FILE}"
    )


# =================================================================================================
# 4. READ ORIGINAL XPT
# =================================================================================================

df = pd.read_sas(
    XPT_FILE,
    format="xport",
    encoding="latin1"
)


#
# Preserve untouched XPT-decoded source.
#

source_df = df.copy()


print("=" * 110)

print(
    "NHANES 2015-2016 SLQ_I"
)

print(
    "SLEEP DISORDERS QUESTIONNAIRE"
)

print("=" * 110)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Variables:"
)


print(
    df.columns.tolist()
)


print(
    "\nSource pandas dtypes:"
)


print(
    df.dtypes
)


# =================================================================================================
# 5. EXPECTED SOURCE STRUCTURE
# =================================================================================================

EXPECTED_COLUMNS = [

    "SEQN",

    "SLQ300",

    "SLQ310",

    "SLD012",

    "SLQ030",

    "SLQ040",

    "SLQ050",

    "SLQ120"

]


EXPECTED_ROWS = 6327


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SLQ_I structure differs from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected SLQ_I row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(df):,}"
    )


print(
    "PASS: SLQ_I structure = 6,327 rows x 8 columns."
)


# =================================================================================================
# 6. DEFINE FIELD GROUPS
# =================================================================================================

CHARACTER_COLUMNS = [

    "SLQ300",

    "SLQ310"

]


DECIMAL_COLUMNS = [

    "SLD012"

]


INTEGER_COLUMNS = [

    "SEQN",

    "SLQ030",

    "SLQ040",

    "SLQ050",

    "SLQ120"

]


NUMERIC_COLUMNS = [

    "SEQN",

    "SLD012",

    "SLQ030",

    "SLQ040",

    "SLQ050",

    "SLQ120"

]


print(
    "\n--- FIELD TYPE PLAN ---"
)


print(
    "Character fields:",
    CHARACTER_COLUMNS
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "Integral/code fields:",
    INTEGER_COLUMNS
)


# =================================================================================================
# 7. HELPER - CHARACTER TEXT VIEW
# =================================================================================================

#
# IMPORTANT:
#
# This helper creates a text VIEW for validation.
#
# It does NOT change the original source dataframe.
#
# It works whether pandas loaded a character column as:
#
#   object
#
# or:
#
#   string / StringDtype
#

def text_view(series):

    return (
        series
        .astype("string")
        .fillna("")
    )


# =================================================================================================
# 8. VALIDATE NUMERIC SOURCE FIELDS
# =================================================================================================

for col in NUMERIC_COLUMNS:

    if not pd.api.types.is_numeric_dtype(
        df[col]
    ):

        raise ValueError(
            f"{col} expected to be numeric but loaded as "
            f"{df[col].dtype}."
        )


print(
    "PASS: Numeric source fields validated."
)


# =================================================================================================
# 9. VALIDATE CHARACTER CONTENT
# =================================================================================================

#
# DO NOT use:
#
#     is_object_dtype()
#
# because pandas may use StringDtype.
#
# Instead validate actual text content.
#

for col in CHARACTER_COLUMNS:

    text = text_view(
        df[col]
    )


    #
    # Valid values are:
    #
    #   HH:MM
    #   77777
    #   99999
    #   blank
    #

    time_mask = text.str.fullmatch(
        r"\d{2}:\d{2}",
        na=False
    )


    special_mask = text.isin(
        [
            "77777",
            "99999",
            ""
        ]
    )


    invalid_mask = ~(
        time_mask
        |
        special_mask
    )


    invalid_count = int(
        invalid_mask.sum()
    )


    if invalid_count != 0:

        print(
            f"\nInvalid examples for {col}:"
        )


        print(
            text.loc[
                invalid_mask
            ]
            .head(20)
            .tolist()
        )


        raise ValueError(
            f"{col} contains {invalid_count:,} "
            "unexpected character values."
        )


    #
    # Verify valid HH:MM clock ranges.
    #

    actual_times = (
        text.loc[
            time_mask
        ]
    )


    hours = pd.to_numeric(

        actual_times
        .str.slice(
            0,
            2
        ),

        errors="raise"

    )


    minutes = pd.to_numeric(

        actual_times
        .str.slice(
            3,
            5
        ),

        errors="raise"

    )


    invalid_clock = (

        ~hours.between(
            0,
            23
        )

        |

        ~minutes.between(
            0,
            59
        )

    )


    if invalid_clock.any():

        raise ValueError(
            f"{col} contains invalid HH:MM clock values."
        )


    print(
        f"PASS: {col} character content validated "
        f"independent of pandas dtype."
    )


# =================================================================================================
# 10. SEQN SOURCE CHECK
# =================================================================================================

seqn_min = int(
    df["SEQN"].min()
)


seqn_max = int(
    df["SEQN"].max()
)


seqn_missing = int(

    df["SEQN"]
    .isna()
    .sum()

)


seqn_duplicates = int(

    df["SEQN"]
    .duplicated()
    .sum()

)


print(
    "\n--- SEQN SOURCE CHECK ---"
)


print(
    "Minimum:",
    seqn_min
)


print(
    "Maximum:",
    seqn_max
)


print(
    "Missing:",
    seqn_missing
)


print(
    "Duplicates:",
    seqn_duplicates
)


if seqn_min != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if seqn_max != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if seqn_missing != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if seqn_duplicates != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN range 83732-93702 validated."
)


# =================================================================================================
# 11. DEFINE KNOWN SAS XPORT TINY VALUE
# =================================================================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


# =================================================================================================
# 12. FIND TINY VALUES
# =================================================================================================

tiny_counts = (

    df[NUMERIC_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPORT TINY-ZERO CHECK ---"
)


print(
    tiny_counts.to_string()
)


print(
    "\nTotal tiny XPORT values:",
    f"{tiny_total:,}"
)


# =================================================================================================
# 13. VALIDATED TINY-ZERO PATTERN
# =================================================================================================

EXPECTED_TINY_COUNTS = {

    "SLQ030": 1820,

    "SLQ040": 4535,

    "SLQ120": 1159

}


actual_tiny_counts = (

    tiny_counts[
        tiny_counts > 0
    ]

    .astype(int)

    .to_dict()

)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: SLQ_I tiny-value pattern differs "
        "from validated source.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 7514:

    raise ValueError(
        "Unexpected total tiny-value count.\n"
        f"Expected: 7,514\n"
        f"Actual:   {tiny_total:,}"
    )


print(
    "PASS: Exact 7,514-value tiny-zero pattern validated."
)


# =================================================================================================
# 14. ORDINARY ZERO CHECK BEFORE REPAIR
# =================================================================================================

ordinary_zero_counts = (

    df[NUMERIC_COLUMNS]

    .eq(0)

    .sum()

)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORDINARY ZERO CHECK BEFORE REPAIR ---"
)


print(
    ordinary_zero_counts.to_string()
)


print(
    "\nTotal ordinary zeros:",
    ordinary_zero_total
)


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before XPORT correction."
    )


print(
    "PASS: Legitimate zero categories are represented "
    "by the XPORT tiny value."
)


# =================================================================================================
# 15. BUILD EXPECTED CORRECTED SOURCE
# =================================================================================================

expected_df = (
    source_df.copy()
)


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[

        expected_df[col]
        .eq(TINY_VALUE),

        col

    ] = 0


# =================================================================================================
# 16. RESTORE ONLY VALIDATED TRUE ZEROS
# =================================================================================================

correction_counts = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: tiny-value count changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_counts[
        col
    ] = actual_count


total_corrections = sum(
    correction_counts.values()
)


print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, count in correction_counts.items():

    print(
        f"{col}: {count:,}"
    )


print(
    "Total:",
    f"{total_corrections:,}"
)


if total_corrections != 7514:

    raise ValueError(
        "Incorrect zero-restoration total."
    )


print(
    "PASS: 7,514 validated zeros restored."
)


# =================================================================================================
# 17. VERIFY NO TINY VALUE REMAINS
# =================================================================================================

remaining_tiny = int(

    df[NUMERIC_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

    .sum()

)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# =================================================================================================
# 18. VERIFY CORRECTED TRUE-ZERO PATTERN
# =================================================================================================

corrected_zero_counts = {}


for col in NUMERIC_COLUMNS:

    count = int(

        df[col]
        .eq(0)
        .sum()

    )


    if count > 0:

        corrected_zero_counts[
            col
        ] = count


print(
    "\n--- CORRECTED TRUE-ZERO DISTRIBUTION ---"
)


print(
    corrected_zero_counts
)


if corrected_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Corrected zero distribution differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{corrected_zero_counts}"
    )


print(
    "PASS: Corrected zero distribution validated."
)


# =================================================================================================
# 19. CDC CHECK - SLQ300
# =================================================================================================

slq300 = text_view(
    df["SLQ300"]
)


slq300_time = int(

    slq300
    .str.fullmatch(
        r"\d{2}:\d{2}",
        na=False
    )
    .sum()

)


slq300_refused = int(

    slq300
    .eq("77777")
    .sum()

)


slq300_dontknow = int(

    slq300
    .eq("99999")
    .sum()

)


slq300_missing = int(

    slq300
    .eq("")
    .sum()

)


print(
    "\n--- CDC CHECK: SLQ300 ---"
)


print(
    "Recorded HH:MM:",
    f"{slq300_time:,}"
)


print(
    "Refused:",
    slq300_refused
)


print(
    "Don't know:",
    slq300_dontknow
)


print(
    "Missing:",
    slq300_missing
)


if slq300_time != 6303:

    raise ValueError(
        "SLQ300 recorded-time count differs from CDC."
    )


if slq300_refused != 0:

    raise ValueError(
        "SLQ300 Refused count differs from CDC."
    )


if slq300_dontknow != 20:

    raise ValueError(
        "SLQ300 Don't know count differs from CDC."
    )


if slq300_missing != 4:

    raise ValueError(
        "SLQ300 missing count differs from CDC."
    )


print(
    "PASS: SLQ300 matches CDC."
)


# =================================================================================================
# 20. CDC CHECK - SLQ310
# =================================================================================================

slq310 = text_view(
    df["SLQ310"]
)


slq310_time = int(

    slq310
    .str.fullmatch(
        r"\d{2}:\d{2}",
        na=False
    )
    .sum()

)


slq310_refused = int(

    slq310
    .eq("77777")
    .sum()

)


slq310_dontknow = int(

    slq310
    .eq("99999")
    .sum()

)


slq310_missing = int(

    slq310
    .eq("")
    .sum()

)


print(
    "\n--- CDC CHECK: SLQ310 ---"
)


print(
    "Recorded HH:MM:",
    f"{slq310_time:,}"
)


print(
    "Refused:",
    slq310_refused
)


print(
    "Don't know:",
    slq310_dontknow
)


print(
    "Missing:",
    slq310_missing
)


if slq310_time != 6307:

    raise ValueError(
        "SLQ310 recorded-time count differs from CDC."
    )


if slq310_refused != 0:

    raise ValueError(
        "SLQ310 Refused count differs from CDC."
    )


if slq310_dontknow != 16:

    raise ValueError(
        "SLQ310 Don't know count differs from CDC."
    )


if slq310_missing != 4:

    raise ValueError(
        "SLQ310 missing count differs from CDC."
    )


print(
    "PASS: SLQ310 matches CDC."
)


# =================================================================================================
# 21. CDC CHECK - SLD012
# =================================================================================================

sld012_nonmissing = int(

    df["SLD012"]
    .notna()
    .sum()

)


sld012_missing = int(

    df["SLD012"]
    .isna()
    .sum()

)


sld012_min = float(
    df["SLD012"].min()
)


sld012_max = float(
    df["SLD012"].max()
)


print(
    "\n--- CDC CHECK: SLD012 ---"
)


print(
    "Nonmissing:",
    f"{sld012_nonmissing:,}"
)


print(
    "Missing:",
    sld012_missing
)


print(
    "Range:",
    sld012_min,
    "to",
    sld012_max
)


if sld012_nonmissing != 6294:

    raise ValueError(
        "SLD012 nonmissing count differs from CDC."
    )


if sld012_missing != 33:

    raise ValueError(
        "SLD012 missing count differs from CDC."
    )


if sld012_min != 2.0:

    raise ValueError(
        "SLD012 minimum differs from CDC."
    )


if sld012_max != 14.5:

    raise ValueError(
        "SLD012 maximum differs from CDC."
    )


#
# Verify all values are in half-hour increments.
#

sld012_values = (

    df["SLD012"]

    .dropna()

    .to_numpy(
        dtype=float
    )

)


half_hour_valid = np.isclose(

    sld012_values * 2,

    np.round(
        sld012_values * 2
    ),

    rtol=0,

    atol=1e-12

)


if not half_hour_valid.all():

    raise ValueError(
        "SLD012 contains values outside half-hour increments."
    )


print(
    "PASS: SLD012 preserved as genuine half-hour decimal data."
)


# =================================================================================================
# 22. CDC CHECK - SLQ030
# =================================================================================================

EXPECTED_SLQ030 = {

    0.0: 1820,

    1.0: 1475,

    2.0: 1067,

    3.0: 1513,

    7.0: 2,

    9.0: 450

}


actual_slq030 = (

    df["SLQ030"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


if actual_slq030 != EXPECTED_SLQ030:

    raise ValueError(
        "SLQ030 distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_SLQ030}\n\n"
        f"Actual:\n{actual_slq030}"
    )


if int(
    df["SLQ030"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SLQ030 missing count differs from CDC."
    )


print(
    "PASS: SLQ030 matches CDC."
)


# =================================================================================================
# 23. CDC CHECK - SLQ040
# =================================================================================================

EXPECTED_SLQ040 = {

    0.0: 4535,

    1.0: 773,

    2.0: 351,

    3.0: 286,

    7.0: 2,

    9.0: 380

}


actual_slq040 = (

    df["SLQ040"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


if actual_slq040 != EXPECTED_SLQ040:

    raise ValueError(
        "SLQ040 distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_SLQ040}\n\n"
        f"Actual:\n{actual_slq040}"
    )


if int(
    df["SLQ040"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SLQ040 missing count differs from CDC."
    )


print(
    "PASS: SLQ040 matches CDC."
)


# =================================================================================================
# 24. CDC CHECK - SLQ050
# =================================================================================================

EXPECTED_SLQ050 = {

    1.0: 1629,

    2.0: 4694,

    9.0: 4

}


actual_slq050 = (

    df["SLQ050"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


if actual_slq050 != EXPECTED_SLQ050:

    raise ValueError(
        "SLQ050 distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_SLQ050}\n\n"
        f"Actual:\n{actual_slq050}"
    )


if int(
    df["SLQ050"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SLQ050 missing count differs from CDC."
    )


print(
    "PASS: SLQ050 matches CDC."
)


# =================================================================================================
# 25. CDC CHECK - SLQ120
# =================================================================================================

EXPECTED_SLQ120 = {

    0.0: 1159,

    1.0: 1454,

    2.0: 2053,

    3.0: 1131,

    4.0: 525,

    9.0: 5

}


actual_slq120 = (

    df["SLQ120"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


if actual_slq120 != EXPECTED_SLQ120:

    raise ValueError(
        "SLQ120 distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_SLQ120}\n\n"
        f"Actual:\n{actual_slq120}"
    )


if int(
    df["SLQ120"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SLQ120 missing count differs from CDC."
    )


print(
    "PASS: SLQ120 matches CDC."
)


# =================================================================================================
# 26. FRACTIONAL-VALUE DISCOVERY
# =================================================================================================

fractional_counts = {}


for col in NUMERIC_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) == 0:

        count = 0


    else:

        count = int(

            (
                ~np.isclose(

                    values,

                    np.round(
                        values
                    ),

                    rtol=0,

                    atol=1e-12

                )
            )

            .sum()

        )


    fractional_counts[
        col
    ] = count


print(
    "\n--- FRACTIONAL VALUE DISCOVERY ---"
)


for col, count in fractional_counts.items():

    print(
        f"{col}: {count:,}"
    )


#
# Actual attachment:
#
# SLD012 has 2,090 fractional observations.
#

if fractional_counts["SLD012"] != 2090:

    raise ValueError(
        "Unexpected SLD012 fractional-value count."
    )


for col in INTEGER_COLUMNS:

    if fractional_counts[col] != 0:

        raise ValueError(
            f"{col} unexpectedly contains fractional values."
        )


print(
    "PASS: SLD012 is the only genuine fractional numeric field."
)


# =================================================================================================
# 27. VERIFY ONLY VALIDATED NUMERIC VALUES CHANGED
# =================================================================================================

source_change_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        source_change_failures.append(
            col
        )


if source_change_failures:

    raise ValueError(
        "Unexpected numeric source changes:\n"
        + str(
            source_change_failures
        )
    )


print(
    "PASS: Only validated tiny-zero values changed."
)


# =================================================================================================
# 28. CHARACTER SOURCE FIDELITY BEFORE EXPORT
# =================================================================================================

for col in CHARACTER_COLUMNS:

    original_text = text_view(
        source_df[col]
    )


    current_text = text_view(
        df[col]
    )


    if not original_text.equals(
        current_text
    ):

        raise ValueError(
            f"{col} character values changed during cleaning."
        )


print(
    "PASS: SLQ300 / SLQ310 remain unchanged."
)


# =================================================================================================
# 29. MISSING-POSITION FIDELITY
# =================================================================================================

#
# Numeric variables:
#
# missing = NaN
#

for col in NUMERIC_COLUMNS:

    source_missing = (

        source_df[col]
        .isna()
        .to_numpy()
    )


    current_missing = (

        df[col]
        .isna()
        .to_numpy()
    )


    if not np.array_equal(

        source_missing,

        current_missing

    ):

        raise ValueError(
            f"{col}: numeric missing positions changed."
        )


#
# Character variables:
#
# normalize both NaN and blank to "" for comparison.
#

for col in CHARACTER_COLUMNS:

    source_missing = (

        text_view(
            source_df[col]
        )

        .eq("")

        .to_numpy()

    )


    current_missing = (

        text_view(
            df[col]
        )

        .eq("")

        .to_numpy()

    )


    if not np.array_equal(

        source_missing,

        current_missing

    ):

        raise ValueError(
            f"{col}: character missing positions changed."
        )


print(
    "PASS: Missing positions preserved."
)


# =================================================================================================
# 30. CONVERT ONLY PROVEN INTEGRAL/CODE VARIABLES
# =================================================================================================

#
# DO NOT convert:
#
#   SLD012
#
# DO NOT touch:
#
#   SLQ300
#   SLQ310
#

for col in INTEGER_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) > 0:

        integral_mask = np.isclose(

            values,

            np.round(
                values
            ),

            rtol=0,

            atol=1e-12

        )


        if not integral_mask.all():

            raise ValueError(
                f"{col} contains unexpected fractional values."
            )


    df[col] = (

        df[col]

        .astype("Int64")

    )


print(
    "PASS: Only proven integral/code fields converted to Int64."
)


# =================================================================================================
# 31. PRE-EXPORT NUMERIC FIDELITY
# =================================================================================================

pre_export_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        pd.to_numeric(
            df[col],
            errors="coerce"
        )

        .to_numpy(
            dtype=float,
            na_value=np.nan
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export numeric fidelity failures:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric fidelity."
)


# =================================================================================================
# 32. PRE-EXPORT CHARACTER FIDELITY
# =================================================================================================

for col in CHARACTER_COLUMNS:

    if not text_view(
        df[col]
    ).equals(
        text_view(
            source_df[col]
        )
    ):

        raise ValueError(
            f"{col} changed before export."
        )


print(
    "PASS: Pre-export character fidelity."
)


# =================================================================================================
# 33. EXPORT CORRECTED CSV
# =================================================================================================

#
# NO:
#
#     float_format=
#
# NO:
#
#     round()
#
# SLD012 remains genuine decimal data.
#

df.to_csv(

    CSV_FILE,

    index=False,

    na_rep=""

)


print(
    "\nCorrected CSV created:"
)


print(
    CSV_FILE
)


# =================================================================================================
# 34. RE-READ CSV AS TEXT
# =================================================================================================

raw_csv = pd.read_csv(

    CSV_FILE,

    dtype=str,

    keep_default_na=False

)


if raw_csv.shape != (

    EXPECTED_ROWS,

    len(EXPECTED_COLUMNS)

):

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    raw_csv.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column order changed."
    )


print(
    "PASS: Final CSV structure preserved."
)


# =================================================================================================
# 35. CHARACTER TEXT FIDELITY AFTER EXPORT
# =================================================================================================

for col in CHARACTER_COLUMNS:

    source_text = (

        text_view(
            source_df[col]
        )

        .to_numpy(
            dtype=object
        )

    )


    final_text = (

        raw_csv[col]

        .to_numpy(
            dtype=object
        )

    )


    if not np.array_equal(

        source_text,

        final_text

    ):

        mismatch_mask = (

            source_text
            !=
            final_text
        )


        mismatch_count = int(
            mismatch_mask.sum()
        )


        print(
            f"{col} text mismatches:",
            mismatch_count
        )


        raise ValueError(
            f"{col} changed during CSV export."
        )


print(
    "PASS: SLQ300 / SLQ310 preserved text-for-text."
)


# =================================================================================================
# 36. NUMERIC XPT -> CSV ROUND-TRIP FIDELITY
# =================================================================================================

roundtrip_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    final_values = (

        pd.to_numeric(

            raw_csv[col]
            .where(
                raw_csv[col] != "",
                np.nan
            ),

            errors="coerce"

        )

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        final_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity failures:\n"
        + str(
            roundtrip_failures
        )
    )


print(
    "PASS: Numeric XPT -> CSV round-trip fidelity."
)


# =================================================================================================
# 37. CHECK UNWANTED ".0"
# =================================================================================================

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    count = int(

        (
            raw_csv[col]
            .ne("")

            &

            raw_csv[col]
            .str.endswith(".0")
        )

        .sum()

    )


    if count > 0:

        dot_zero_errors[
            col
        ] = count


if dot_zero_errors:

    raise ValueError(
        "Unnecessary '.0' formatting remains:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in integral/code fields."
)


# =================================================================================================
# 38. FINAL SEQN CHECK
# =================================================================================================

if (
    raw_csv["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    raw_csv["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV contains SEQN '.0'."
    )


if (

    ~raw_csv["SEQN"]
    .str.fullmatch(r"\d+")

).any():

    raise ValueError(
        "Final CSV contains invalid SEQN."
    )


final_seqn = pd.to_numeric(

    raw_csv["SEQN"],

    errors="raise"

)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


print(
    "PASS: Final SEQN clean."
)


# =================================================================================================
# 39. FINAL SLD012 DECIMAL CHECK
# =================================================================================================

final_sld012 = pd.to_numeric(

    raw_csv["SLD012"]
    .where(
        raw_csv["SLD012"] != "",
        np.nan
    ),

    errors="raise"

)


if int(
    final_sld012.notna().sum()
) != 6294:

    raise ValueError(
        "Final SLD012 nonmissing count differs."
    )


if float(
    final_sld012.min()
) != 2.0:

    raise ValueError(
        "Final SLD012 minimum differs."
    )


if float(
    final_sld012.max()
) != 14.5:

    raise ValueError(
        "Final SLD012 maximum differs."
    )


fractional_final = (

    final_sld012
    .dropna()
    .to_numpy(
        dtype=float
    )

)


fractional_count = int(

    (
        ~np.isclose(

            fractional_final,

            np.round(
                fractional_final
            ),

            rtol=0,

            atol=1e-12

        )
    )

    .sum()

)


if fractional_count != 2090:

    raise ValueError(
        "SLD012 fractional values were altered."
    )


print(
    "PASS: SLD012 decimal values preserved."
)


# =================================================================================================
# 40. FINAL TINY-XPORT CHECK
# =================================================================================================

final_tiny_total = 0


for col in NUMERIC_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_tiny_total += int(

        values
        .eq(TINY_VALUE)
        .sum()

    )


if final_tiny_total != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# =================================================================================================
# 41. FINAL TRUE-ZERO DISTRIBUTION
# =================================================================================================

final_zero_counts = {}


for col in NUMERIC_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    count = int(

        values
        .eq(0)
        .sum()

    )


    if count > 0:

        final_zero_counts[
            col
        ] = count


print(
    "\n--- FINAL TRUE-ZERO DISTRIBUTION ---"
)


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero distribution differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Exactly 7,514 validated true zeros remain."
)


# =================================================================================================
# 42. FINAL SCIENTIFIC-NOTATION CHECK
# =================================================================================================

scientific_notation_fields = {}


for col in NUMERIC_COLUMNS:

    count = int(

        raw_csv[col]

        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_notation_fields[
            col
        ] = count


if scientific_notation_fields:

    raise ValueError(
        "Unexpected scientific notation remains:\n"
        + str(
            scientific_notation_fields
        )
    )


print(
    "PASS: No unexpected scientific notation."
)


# =================================================================================================
# 43. FINAL MISSING-POSITION CHECK
# =================================================================================================

for col in NUMERIC_COLUMNS:

    source_missing = (

        source_df[col]
        .isna()
        .to_numpy()

    )


    final_missing = (

        raw_csv[col]
        .eq("")
        .to_numpy()

    )


    if not np.array_equal(

        source_missing,

        final_missing

    ):

        raise ValueError(
            f"{col}: final numeric missing positions differ."
        )


for col in CHARACTER_COLUMNS:

    source_missing = (

        text_view(
            source_df[col]
        )

        .eq("")

        .to_numpy()

    )


    final_missing = (

        raw_csv[col]
        .eq("")
        .to_numpy()

    )


    if not np.array_equal(

        source_missing,

        final_missing

    ):

        raise ValueError(
            f"{col}: final character missing positions differ."
        )


print(
    "PASS: Missing positions preserved."
)


# =================================================================================================
# 44. FINAL CDC DISTRIBUTION RECHECKS
# =================================================================================================

EXPECTED_FINAL_SLQ030 = {

    "0": 1820,

    "1": 1475,

    "2": 1067,

    "3": 1513,

    "7": 2,

    "9": 450

}


EXPECTED_FINAL_SLQ040 = {

    "0": 4535,

    "1": 773,

    "2": 351,

    "3": 286,

    "7": 2,

    "9": 380

}


EXPECTED_FINAL_SLQ050 = {

    "1": 1629,

    "2": 4694,

    "9": 4

}


EXPECTED_FINAL_SLQ120 = {

    "0": 1159,

    "1": 1454,

    "2": 2053,

    "3": 1131,

    "4": 525,

    "9": 5

}


for col, expected in {

    "SLQ030":
        EXPECTED_FINAL_SLQ030,

    "SLQ040":
        EXPECTED_FINAL_SLQ040,

    "SLQ050":
        EXPECTED_FINAL_SLQ050,

    "SLQ120":
        EXPECTED_FINAL_SLQ120

}.items():

    actual = (

        raw_csv[col]

        .value_counts(
            dropna=False
        )

        .to_dict()

    )


    if actual != expected:

        raise ValueError(
            f"Final {col} distribution differs.\n\n"
            f"Expected:\n{expected}\n\n"
            f"Actual:\n{actual}"
        )


    print(
        f"PASS: Final {col} matches CDC."
    )


# =================================================================================================
# 45. FINAL CHARACTER FREQUENCY RECHECK
# =================================================================================================

final_slq300 = (
    raw_csv["SLQ300"]
)


if int(

    final_slq300
    .str.fullmatch(
        r"\d{2}:\d{2}",
        na=False
    )
    .sum()

) != 6303:

    raise ValueError(
        "Final SLQ300 recorded-time count differs."
    )


if int(
    final_slq300.eq("99999").sum()
) != 20:

    raise ValueError(
        "Final SLQ300 Don't know count differs."
    )


if int(
    final_slq300.eq("").sum()
) != 4:

    raise ValueError(
        "Final SLQ300 missing count differs."
    )


final_slq310 = (
    raw_csv["SLQ310"]
)


if int(

    final_slq310
    .str.fullmatch(
        r"\d{2}:\d{2}",
        na=False
    )
    .sum()

) != 6307:

    raise ValueError(
        "Final SLQ310 recorded-time count differs."
    )


if int(
    final_slq310.eq("99999").sum()
) != 16:

    raise ValueError(
        "Final SLQ310 Don't know count differs."
    )


if int(
    final_slq310.eq("").sum()
) != 4:

    raise ValueError(
        "Final SLQ310 missing count differs."
    )


print(
    "PASS: Final SLQ300 / SLQ310 frequencies match CDC."
)


# =================================================================================================
# 46. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2015-2016 SLQ_I CONVERSION COMPLETE"
)


print(
    "=" * 110
)


print(
    "Component:"
)


print(
    "  Questionnaire"
)


print(
    ""
)


print(
    "Section:"
)


print(
    "  Sleep Disorders"
)


print(
    ""
)


print(
    "Cycle:"
)


print(
    "  2015-2016"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(raw_csv):,}"
)


print(
    "Columns:",
    len(raw_csv.columns)
)


print(
    "Unique SEQN:",
    f"{raw_csv['SEQN'].nunique():,}"
)


print(
    "SEQN range:"
)


print(
    "  83732-93702"
)


print(
    ""
)


print(
    "Validated XPORT zero restorations:"
)


print(
    "  SLQ030 = 1,820"
)


print(
    "  SLQ040 = 4,535"
)


print(
    "  SLQ120 = 1,159"
)


print(
    "  ----------------"
)


print(
    "  TOTAL   = 7,514"
)


print(
    ""
)


print(
    "Character variables:"
)


print(
    "  SLQ300"
)


print(
    "  SLQ310"
)


print(
    "  Fidelity validated by CONTENT, not pandas dtype."
)


print(
    ""
)


print(
    "Genuine decimal variable:"
)


print(
    "  SLD012"
)


print(
    "  Nonmissing = 6,294"
)


print(
    "  Range      = 2.0-14.5"
)


print(
    "  Fractional observations = 2,090"
)


print(
    ""
)


print(
    "Integral/code variables:"
)


print(
    "  SEQN"
)


print(
    "  SLQ030"
)


print(
    "  SLQ040"
)


print(
    "  SLQ050"
)


print(
    "  SLQ120"
)


print(
    ""
)


print(
    "SLQ030=0 / Never:"
)


print(
    "  1,820"
)


print(
    "SLQ040=0 / Never:"
)


print(
    "  4,535"
)


print(
    "SLQ120=0 / Never:"
)


print(
    "  1,159"
)


print(
    ""
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate SEQN: 0"
)


print(
    "Remaining tiny XPORT artifacts: 0"
)


print(
    "Character text changes: 0"
)


print(
    "SLD012 integerization: 0"
)


print(
    "Missing-position changes: 0"
)


print(
    "Scientific notation: 0"
)


print(
    "Numeric XPT -> CSV fidelity: PASS"
)


print(
    "Character source -> CSV fidelity: PASS"
)


print(
    "=" * 110
)

NHANES 2015-2016 SLQ_I
SLEEP DISORDERS QUESTIONNAIRE
Rows: 6,327
Columns: 8
Variables:
['SEQN', 'SLQ300', 'SLQ310', 'SLD012', 'SLQ030', 'SLQ040', 'SLQ050', 'SLQ120']

Source pandas dtypes:
SEQN      float64
SLQ300        str
SLQ310        str
SLD012    float64
SLQ030    float64
SLQ040    float64
SLQ050    float64
SLQ120    float64
dtype: object
PASS: SLQ_I structure = 6,327 rows x 8 columns.

--- FIELD TYPE PLAN ---
Character fields: ['SLQ300', 'SLQ310']
Genuine decimal fields: ['SLD012']
Integral/code fields: ['SEQN', 'SLQ030', 'SLQ040', 'SLQ050', 'SLQ120']
PASS: Numeric source fields validated.
PASS: SLQ300 character content validated independent of pandas dtype.
PASS: SLQ310 character content validated independent of pandas dtype.

--- SEQN SOURCE CHECK ---
Minimum: 83732
Maximum: 93702
Missing: 0
Duplicates: 0
PASS: SEQN range 83732-93702 validated.

--- ORIGINAL XPORT TINY-ZERO CHECK ---
SEQN         0
SLD012       0
SLQ030    1820
SLQ040    4535
SLQ050       0
SLQ120    1159

Tot